1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰 수 초과 방지, 답변 생성 시간 단축
3. 임베딩해서 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [ ]:
%pip install -qU  docx2txt langchain-community

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 하나의 청크가 가질 수 있는 토큰 수
    chunk_overlap=200, # 이전 청크와 현재 청크 겹치는 내용 양
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
%pip install -qU langchain-text-splitters

In [3]:
# 임베딩하기
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [ ]:
# 벡터 DB
%pip install -qU langchain langchain-pinecone langchain-openai

In [1]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone

# 1. .env 파일에 있는 변수들을 파이썬 환경 변수로 싹 다 불러오기
load_dotenv()

# 2. 불러온 환경 변수 중에서 Pinecone API 키만 쏙 빼오기
pinecone_api_key = os.environ.get("PINECONE_API_KEY")

# 3. (선택적 방어 로직) 만약 .env 파일에 키를 안 적어뒀다면 에러 띄우기
if not pinecone_api_key:
    raise ValueError(".env 파일에 PINECONE_API_KEY가 세팅되지 않았어!")

# 4. Pinecone 연결 완료!
pc = Pinecone(api_key=pinecone_api_key)

In [10]:
from langchain_pinecone import PineconeVectorStore

index_name='tax-index'
database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)

In [23]:
query = '연봉 5천만원인 직장인의 소득세를 계산해주세요.'
retrieved_docs = database.similarity_search(query, k=5)

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

In [13]:
from langchain_core.prompts import PromptTemplate

template = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question} 
Context: {context} 
Answer:"""

prompt = PromptTemplate.from_template(template)

In [ ]:
prompt

In [14]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 벡터 DB를 retriever 모드로 변환
retriever = database.as_retriever()

# 검색해 온 문서 조각을 긴 텍스트로 이어 붙여주는 헬퍼 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL 문법으로 RAG Chain 조립
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
final_answer = rag_chain.invoke(query)

print("🤖 AI 답변:")
print(final_answer)